# Phase 6 — Production Readiness
**Home Credit Default Risk**

This phase covers the three pillars that separate a competition notebook from a deployable credit risk system:

| Pillar | What it solves |
|---|---|
| **Drift monitoring** | Detect when incoming applicants no longer look like training data |
| **Per-applicant SHAP explanations** | Tell a rejected applicant *why* — required by regulation |
| **Model card** | Document what the model does, its limitations, and fairness checks |

**Inputs required from previous phases:**
- `X_processed` — processed training features (DataFrame with column names)
- `X_test_processed` — processed test features
- `y` — true labels
- `oof_preds` — raw OOF scores
- `test_preds_calibrated` — calibrated test predictions (from Phase 5)
- `fold_models` — list of 5 trained models saved during CV
- `calibrator` — fitted CreditCalibrator from Phase 5

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib
import json
import warnings
from datetime import datetime
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('All imports OK')

---
# PILLAR 1 — Drift Monitoring

**What is drift?**  
When the real-world distribution of incoming applicants shifts away from the training distribution, model performance degrades silently — AUC drops, but you have no labels yet to detect it.  
Drift monitoring catches this early using only feature distributions, no labels needed.

**Two types:**
- **Feature drift** — input features look different (applicant population changed)
- **Score drift** — model output distribution shifted (caught by PSI on predictions)

**Two metrics we use:**
- **PSI (Population Stability Index)** — industry standard for score monitoring
- **KS statistic** — per-feature distribution test

## 1.1 PSI — Population Stability Index

In [ ]:
def compute_psi(expected: np.ndarray, actual: np.ndarray, n_bins: int = 10) -> float:
    """
    Population Stability Index between a reference (training) and
    current (production) distribution.

    Interpretation:
        PSI < 0.10  → No significant shift, model stable
        PSI 0.10–0.20 → Moderate shift, investigate
        PSI > 0.20  → Major shift, consider retraining
    """
    # Build bins from expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints = np.unique(breakpoints)  # remove duplicates

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts   = np.histogram(actual,   bins=breakpoints)[0]

    # Convert to proportions, add epsilon to avoid log(0)
    eps = 1e-6
    expected_pct = expected_counts / len(expected) + eps
    actual_pct   = actual_counts   / len(actual)   + eps

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return float(psi)


def psi_label(psi: float) -> str:
    if psi < 0.10:  return 'Stable     ✓'
    if psi < 0.20:  return 'Moderate   ⚠️'
    return             'Major drift ❌'


# ── Score-level PSI (train OOF vs test predictions) ──────────────────────────
# In production: expected = training OOF scores, actual = new batch scores
score_psi = compute_psi(oof_preds, test_preds)
print(f'Score PSI (OOF vs test): {score_psi:.4f}  →  {psi_label(score_psi)}')

# ── Feature-level PSI for top features ───────────────────────────────────────
# Pick the top 15 features by importance (or use your SHAP ranking)
top_features = X_processed.columns[:15].tolist()  # replace with your SHAP-ranked list

psi_results = []
for col in top_features:
    if col in X_test_processed.columns:
        psi_val = compute_psi(
            X_processed[col].dropna().values,
            X_test_processed[col].dropna().values
        )
        psi_results.append({'feature': col, 'PSI': psi_val, 'status': psi_label(psi_val)})

psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
print(f'\nFeature PSI Report ({len(psi_df)} features):')
print(psi_df.to_string(index=False))

In [ ]:
# Visualise feature PSI
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#2EA87E' if p < 0.10 else '#F0A500' if p < 0.20 else '#E07B5A'
          for p in psi_df['PSI']]

bars = ax.barh(psi_df['feature'], psi_df['PSI'], color=colors, edgecolor='none', height=0.6)
ax.axvline(0.10, color='#F0A500', linestyle='--', linewidth=1.2, label='Moderate threshold (0.10)')
ax.axvline(0.20, color='#E07B5A', linestyle='--', linewidth=1.2, label='Major drift threshold (0.20)')
ax.set_xlabel('PSI')
ax.set_title('Feature Drift Report — PSI (Train vs Test)')
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('drift_psi.png', bbox_inches='tight')
plt.show()

## 1.2 KS Test — Per-Feature Distribution Check

The **Kolmogorov-Smirnov test** checks whether two samples come from the same distribution.  
A low p-value (< 0.05) means the distributions are significantly different — potential drift.

In [ ]:
def ks_drift_report(train_df: pd.DataFrame, test_df: pd.DataFrame,
                    features: list) -> pd.DataFrame:
    """Run KS test for each feature between train and test distributions."""
    rows = []
    for col in features:
        if col not in test_df.columns:
            continue
        tr = train_df[col].dropna().values
        te = test_df[col].dropna().values
        ks_stat, p_val = stats.ks_2samp(tr, te)
        rows.append({
            'feature':  col,
            'KS stat':  ks_stat,
            'p-value':  p_val,
            'drifted':  '❌ Yes' if p_val < 0.05 else '✓  No',
        })
    return pd.DataFrame(rows).sort_values('KS stat', ascending=False)


ks_df = ks_drift_report(X_processed, X_test_processed, top_features)
print('KS Drift Report:')
print(ks_df.round(5).to_string(index=False))

n_drifted = (ks_df['drifted'] == '❌ Yes').sum()
print(f'\n{n_drifted}/{len(ks_df)} features show significant distribution shift (p < 0.05)')

## 1.3 DriftMonitor — Production Class

In production you call `.check()` on each new batch of applicants.  
It returns a report and raises an alert if drift exceeds thresholds.

In [ ]:
class DriftMonitor:
    """
    Monitors feature and score drift between a reference dataset (training)
    and incoming production batches.

    Usage:
        monitor = DriftMonitor(X_train, oof_scores, features_to_watch)
        monitor.save('drift_monitor.joblib')

        # In production (monthly):
        monitor = DriftMonitor.load('drift_monitor.joblib')
        report  = monitor.check(X_new_batch, new_batch_scores)
    """

    PSI_WARN  = 0.10
    PSI_ALERT = 0.20

    def __init__(self, X_ref: pd.DataFrame, score_ref: np.ndarray,
                 features: list):
        self.features   = features
        self._X_ref     = X_ref[features].copy()
        self._score_ref = score_ref.copy()
        self.fitted_at  = datetime.utcnow().isoformat()

    def check(self, X_new: pd.DataFrame, score_new: np.ndarray,
              batch_label: str = 'current') -> dict:
        """Run full drift check. Returns a structured report dict."""
        report = {
            'batch_label':    batch_label,
            'checked_at':     datetime.utcnow().isoformat(),
            'n_samples':      len(X_new),
            'score_psi':      compute_psi(self._score_ref, score_new),
            'feature_drift':  {},
            'alerts':         [],
        }

        # Score drift
        if report['score_psi'] > self.PSI_ALERT:
            report['alerts'].append(f"ALERT: Score PSI {report['score_psi']:.3f} > {self.PSI_ALERT} — consider retraining")
        elif report['score_psi'] > self.PSI_WARN:
            report['alerts'].append(f"WARN:  Score PSI {report['score_psi']:.3f} > {self.PSI_WARN} — monitor closely")

        # Feature drift
        for col in self.features:
            if col not in X_new.columns:
                continue
            psi = compute_psi(
                self._X_ref[col].dropna().values,
                X_new[col].dropna().values
            )
            ks_stat, p_val = stats.ks_2samp(
                self._X_ref[col].dropna().values,
                X_new[col].dropna().values
            )
            report['feature_drift'][col] = {
                'psi': round(psi, 5), 'ks_stat': round(ks_stat, 5),
                'ks_pvalue': round(p_val, 5),
                'status': psi_label(psi)
            }
            if psi > self.PSI_ALERT:
                report['alerts'].append(f"ALERT: {col} PSI {psi:.3f}")

        return report

    def print_report(self, report: dict):
        print(f"\n{'─'*55}")
        print(f"Drift Report — batch: {report['batch_label']}")
        print(f"Checked at: {report['checked_at']}  |  N={report['n_samples']}")
        print(f"Score PSI: {report['score_psi']:.4f}  →  {psi_label(report['score_psi'])}")
        print(f"{'─'*55}")
        for col, metrics in report['feature_drift'].items():
            print(f"  {col:<35} PSI={metrics['psi']:.4f}  {metrics['status']}")
        if report['alerts']:
            print(f"\n⚡ Alerts ({len(report['alerts'])})")
            for a in report['alerts']:
                print(f"  {a}")
        else:
            print('\n✓ No alerts — model appears stable')
        print(f"{'─'*55}")

    def save(self, path='drift_monitor.joblib'):
        joblib.dump(self, path)
        print(f'DriftMonitor saved → {path}')

    @classmethod
    def load(cls, path='drift_monitor.joblib'):
        obj = joblib.load(path)
        print(f'DriftMonitor loaded ← {path}  (fitted_at={obj.fitted_at})')
        return obj


# Fit and save
monitor = DriftMonitor(
    X_ref=X_processed,
    score_ref=oof_preds,
    features=top_features
)
monitor.save('drift_monitor.joblib')

# Simulate a production check (using test set as the new batch)
report = monitor.check(X_test_processed, test_preds, batch_label='test_batch_2024_01')
monitor.print_report(report)

# Save report as JSON for logging
with open('drift_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nDrift report saved → drift_report.json')

---
# PILLAR 2 — Per-Applicant SHAP Explanations

**Why this matters in credit:**  
Regulations like GDPR (Article 22) and the EU AI Act require that automated credit decisions can be explained to the applicant.  
SHAP gives you the top reasons a specific applicant was declined, ranked by their contribution to the decision.

**What we build here:**
- A `SHAPExplainer` wrapper that takes one applicant's data and returns a human-readable explanation
- A waterfall plot for visual inspection
- A structured JSON output for an API response

## 2.1 Build the Explainer (using last fold model)

In [ ]:
# Use the last fold model — or average SHAP across all folds for more stability
# Here we use the last fold model for simplicity
model     = fold_models[-1]   # last fold LGBMClassifier
explainer = shap.TreeExplainer(model)

# Compute SHAP on a sample of training data for global context
sample_size = min(2000, len(X_processed))
X_sample    = X_processed.sample(n=sample_size, random_state=42)
shap_values = explainer.shap_values(X_sample)

print(f'SHAP values computed on {sample_size} samples')
print(f'Base value (expected output): {explainer.expected_value:.5f}')

## 2.2 Global Summary — What the Model Uses

In [ ]:
# Summary plot: importance + direction in one view
shap.summary_plot(
    shap_values, X_sample,
    plot_type='dot',
    max_display=20,
    show=False
)
plt.title('SHAP Summary — Global Feature Importance', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# Mean absolute SHAP per feature — clean importance ranking
shap_importance = pd.DataFrame({
    'feature':   X_processed.columns,
    'mean_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

print('Top 20 features by mean |SHAP|:')
print(shap_importance.head(20).round(5).to_string(index=False))

## 2.3 SHAPExplainer — Per-Applicant Explanation Wrapper

In [ ]:
class SHAPExplainer:
    """
    Generates human-readable explanations for individual credit decisions.

    Usage:
        exp = SHAPExplainer(fold_models, calibrator)
        exp.save('shap_explainer.joblib')

        # For a single applicant:
        result = exp.explain(applicant_row_df)
        exp.plot_waterfall(result)
        print(exp.format_reasons(result))  # for API / letter
    """

    # Human-readable feature name mapping — extend with your full feature list
    FEATURE_LABELS = {
        'CREDIT_INCOME_RATIO':  'credit amount relative to income',
        'ANNUITY_INCOME_RATIO': 'monthly repayment relative to income',
        'CREDIT_TERM':          'loan repayment term',
        'DAYS_EMPLOYED_RATIO':  'employment stability relative to age',
        'DAYS_EMPLOYED_ANOM':   'employment history availability',
        'AGE_YEARS':            'applicant age',
        'INCOME_PER_PERSON':    'income per household member',
        'GOODS_CREDIT_RATIO':   'goods value relative to credit amount',
        'EXT_SOURCE_1':         'external credit score 1',
        'EXT_SOURCE_2':         'external credit score 2',
        'EXT_SOURCE_3':         'external credit score 3',
        'DAYS_BIRTH':           'applicant age',
        'DAYS_EMPLOYED':        'employment duration',
        'AMT_CREDIT':           'loan amount',
        'AMT_INCOME_TOTAL':     'total annual income',
    }

    def __init__(self, fold_models: list, calibrator):
        # Use last fold model for explanations
        self._explainer  = shap.TreeExplainer(fold_models[-1])
        self._calibrator = calibrator
        self._base_value = self._explainer.expected_value

    def explain(self, X_row: pd.DataFrame, top_n: int = 5) -> dict:
        """
        Explain a single applicant's prediction.

        Args:
            X_row : DataFrame with exactly 1 row (processed features)
            top_n : number of top reasons to return

        Returns:
            dict with raw_score, pd_score, shap_values, top_reasons
        """
        assert len(X_row) == 1, 'Pass exactly one applicant row'

        sv          = self._explainer.shap_values(X_row)[0]   # shape: (n_features,)
        raw_score   = self._base_value + sv.sum()
        pd_score    = float(self._calibrator.predict_proba(np.array([raw_score])))

        # Build sorted reason list
        reasons = pd.DataFrame({
            'feature':     X_row.columns,
            'shap_value':  sv,
            'feature_val': X_row.iloc[0].values,
        })
        reasons['abs_shap'] = reasons['shap_value'].abs()
        reasons = reasons.sort_values('abs_shap', ascending=False).head(top_n)
        reasons['direction'] = reasons['shap_value'].apply(
            lambda x: 'increases risk' if x > 0 else 'decreases risk'
        )
        reasons['label'] = reasons['feature'].map(
            lambda f: self.FEATURE_LABELS.get(f, f.replace('_', ' ').lower())
        )

        return {
            'raw_score':   round(raw_score, 5),
            'pd_score':    round(pd_score, 5),
            'decision':    'DECLINE' if pd_score > 0.15 else 'APPROVE',
            'shap_values': sv,
            'base_value':  self._base_value,
            'top_reasons': reasons.to_dict(orient='records'),
            'feature_names': list(X_row.columns),
        }

    def plot_waterfall(self, result: dict, title: str = 'Why was this decision made?'):
        """Waterfall plot for a single applicant explanation."""
        shap.plots.waterfall(
            shap.Explanation(
                values=result['shap_values'],
                base_values=result['base_value'],
                feature_names=result['feature_names'],
            ),
            max_display=12,
            show=False
        )
        plt.title(f"{title}  |  PD = {result['pd_score']:.1%}  |  {result['decision']}",
                  fontsize=11, pad=10)
        plt.tight_layout()
        plt.savefig('shap_waterfall_applicant.png', bbox_inches='tight')
        plt.show()

    def format_reasons(self, result: dict) -> str:
        """
        Format top reasons as a human-readable string.
        Suitable for a rejection letter or API response.
        """
        lines = [
            f"Decision: {result['decision']}",
            f"Estimated probability of default: {result['pd_score']:.1%}",
            f"\nTop factors influencing this decision:",
        ]
        for i, r in enumerate(result['top_reasons'], 1):
            direction = '↑ Increases risk' if r['shap_value'] > 0 else '↓ Decreases risk'
            lines.append(f"  {i}. {r['label'].capitalize()} — {direction}")
        return '\n'.join(lines)

    def to_api_response(self, result: dict) -> dict:
        """Structured dict suitable for a REST API JSON response."""
        return {
            'decision':           result['decision'],
            'probability_default': result['pd_score'],
            'explanation': [
                {
                    'rank':      i + 1,
                    'factor':    r['label'],
                    'direction': r['direction'],
                    'impact':    round(abs(r['shap_value']), 5),
                }
                for i, r in enumerate(result['top_reasons'])
            ]
        }

    def save(self, path='shap_explainer.joblib'):
        joblib.dump(self, path)
        print(f'SHAPExplainer saved → {path}')

    @classmethod
    def load(cls, path='shap_explainer.joblib'):
        obj = joblib.load(path)
        print(f'SHAPExplainer loaded ← {path}')
        return obj


# Build and save
explainer_wrapper = SHAPExplainer(fold_models, calibrator)
explainer_wrapper.save('shap_explainer.joblib')
print('SHAPExplainer ready')

## 2.4 Explain a Single Applicant

In [ ]:
# Pick the applicant with the highest predicted default risk
highest_risk_idx = test_preds_calibrated.argmax()
applicant        = X_test_processed.iloc[[highest_risk_idx]]

print(f'Explaining applicant index {highest_risk_idx}')
print(f'Calibrated PD score: {test_preds_calibrated[highest_risk_idx]:.4f}\n')

result = explainer_wrapper.explain(applicant, top_n=5)

# 1. Waterfall plot
explainer_wrapper.plot_waterfall(result)

# 2. Human-readable reasons
print(explainer_wrapper.format_reasons(result))

# 3. API-ready JSON
api_response = explainer_wrapper.to_api_response(result)
print('\nAPI response (JSON):')
print(json.dumps(api_response, indent=2))

In [ ]:
# Also explain a low-risk applicant for comparison
lowest_risk_idx = test_preds_calibrated.argmin()
applicant_low   = X_test_processed.iloc[[lowest_risk_idx]]

result_low = explainer_wrapper.explain(applicant_low, top_n=5)
explainer_wrapper.plot_waterfall(result_low, title='Why was this applicant approved?')
print(explainer_wrapper.format_reasons(result_low))

---
# PILLAR 3 — Model Card

A **model card** is a short document that records:
- What the model does and how it was built
- Its performance and known limitations
- Fairness considerations
- How it should and should not be used

Originally proposed by Google (Mitchell et al., 2019). Now standard practice in regulated industries and increasingly required by the EU AI Act for high-risk AI systems (credit scoring is explicitly listed as high-risk).

## 3.1 Fairness Check — Performance Across Groups

Before writing the model card, check whether the model performs equally across demographic groups.  
We check AUC by gender and age group — two of the most common fairness concerns in credit.

In [ ]:
# Load original training data for demographic features
# train = pd.read_csv('application_train.csv')  # if not in scope

fairness_df = pd.DataFrame({
    'oof_pred': oof_preds,
    'target':   y,
})

# Add demographic columns from original train (before pipeline processing)
if 'CODE_GENDER' in train.columns:
    fairness_df['gender'] = train['CODE_GENDER'].values

if 'DAYS_BIRTH' in train.columns:
    fairness_df['age_group'] = pd.cut(
        -train['DAYS_BIRTH'] / 365,
        bins=[18, 30, 40, 50, 60, 100],
        labels=['18-30', '31-40', '41-50', '51-60', '60+']
    ).values

# AUC by gender
if 'gender' in fairness_df.columns:
    print('AUC by gender:')
    for gender, grp in fairness_df[fairness_df['gender'].isin(['M', 'F'])].groupby('gender'):
        auc = roc_auc_score(grp['target'], grp['oof_pred'])
        n   = len(grp)
        dr  = grp['target'].mean()
        print(f'  {gender}: AUC={auc:.5f}  n={n:,}  default_rate={dr:.3f}')

# AUC by age group
if 'age_group' in fairness_df.columns:
    print('\nAUC by age group:')
    for age_grp, grp in fairness_df.groupby('age_group', observed=True):
        if len(grp) > 100 and grp['target'].sum() > 10:
            auc = roc_auc_score(grp['target'], grp['oof_pred'])
            print(f'  {age_grp}: AUC={auc:.5f}  n={len(grp):,}')

## 3.2 Generate and Save the Model Card

In [ ]:
def generate_model_card(
    oof_auc:        float,
    cv_mean:        float,
    cv_std:         float,
    ensemble_auc:   float,
    n_features:     int,
    n_train:        int,
    default_rate:   float,
    feature_list:   list,
) -> str:
    """Generate a Markdown model card."""
    date = datetime.utcnow().strftime('%Y-%m-%d')

    return f"""# Model Card — Home Credit Default Risk

**Version:** 1.0  
**Date:** {date}  
**Authors:** [Your name]  
**Contact:** [Your email]

---

## Model Details

| Field | Value |
|---|---|
| **Model type** | Stacked ensemble (LightGBM + XGBoost + CatBoost) |
| **Task** | Binary classification — probability of loan default |
| **Output** | Calibrated probability of default (PD) in [0, 1] |
| **Calibration** | Isotonic regression fitted on OOF predictions |
| **Explainability** | Per-applicant SHAP waterfall explanations |

---

## Intended Use

**Primary use:** Assist credit analysts in assessing default risk for consumer loan applications.  
**Intended users:** Credit risk teams, loan officers.  
**Out-of-scope uses:**  
- Final automated rejection without human review  
- Use on applicant populations substantially different from the training data  
- Jurisdictions where algorithmic credit scoring is prohibited  

---

## Training Data

| Field | Value |
|---|---|
| **Source** | Home Credit Group — Kaggle competition dataset |
| **Training samples** | {n_train:,} |
| **Positive rate (default)** | {default_rate:.2%} — class imbalanced |
| **Tables used** | application_train, bureau, bureau_balance, previous_application, POS_CASH_balance, credit_card_balance, installments_payments |
| **Features** | {n_features} engineered features |
| **Target** | 1 = payment difficulty (late >X days on first Y installments), 0 = otherwise |

---

## Performance

All metrics computed on held-out data via stratified 5-fold cross-validation.

| Metric | Value |
|---|---|
| **OOF ROC-AUC** | {oof_auc:.5f} |
| **Mean CV AUC** | {cv_mean:.5f} ± {cv_std:.5f} |
| **Best ensemble AUC** | {ensemble_auc:.5f} |
| **Evaluation metric** | ROC-AUC (competition standard) |

> ⚠️ AUC measures ranking ability, not calibration. Use Brier Score to evaluate probability accuracy.

---

## Top Features

The following features have the highest mean |SHAP| value across the validation set:

{chr(10).join(f'{i+1}. `{f}`' for i, f in enumerate(feature_list[:10]))}

Full feature importance available in `shap_summary.png`.

---

## Fairness & Limitations

- **Class imbalance:** The dataset is ~8% positive. `scale_pos_weight` is used during training to correct for this. Results on the minority class should be interpreted with care.
- **Protected attributes:** `CODE_GENDER` and `DAYS_BIRTH` (age) are present in the data. Performance across demographic groups should be audited before deployment. See fairness check outputs in this notebook.
- **Missing credit history:** ~14% of applicants in training had no bureau history. The model may be less accurate for first-time borrowers.
- **Temporal validity:** This model was trained on data from a specific historical period. Performance may degrade as economic conditions change. PSI monitoring is in place — retrain if score PSI > 0.20.
- **Calibration scope:** The isotonic calibrator was fitted on OOF predictions from this specific dataset. It should be re-fitted if the model or training data changes.

---

## Drift Monitoring

A `DriftMonitor` is fitted on the training distribution and saved as `drift_monitor.joblib`.  
Run monthly on incoming application batches.

| PSI Range | Action |
|---|---|
| < 0.10 | Model stable, no action |
| 0.10 – 0.20 | Investigate feature shifts |
| > 0.20 | Retrain model |

---

## Ethical Considerations

Credit scoring directly affects people's access to financial services. This model should:

1. Always be reviewed by a human analyst before a final rejection decision
2. Provide SHAP-based explanations to declined applicants upon request (GDPR Article 22)
3. Be audited annually for demographic parity and equalised odds across protected groups
4. Not be used as the sole determinant of creditworthiness

---

## Artefacts

| File | Description |
|---|---|
| `credit_calibrator.joblib` | Fitted isotonic calibrator |
| `shap_explainer.joblib` | Per-applicant SHAP explainer |
| `drift_monitor.joblib` | Fitted drift monitor |
| `phase5_calibrated_submission.csv` | Final calibrated predictions |
| `shap_summary.png` | Global feature importance |
| `drift_psi.png` | Feature drift report |
"""


# Generate
model_card = generate_model_card(
    oof_auc      = roc_auc_score(y, oof_preds),
    cv_mean      = np.mean(fold_aucs),           # from your CV loop
    cv_std       = np.std(fold_aucs),
    ensemble_auc = 0.792610,                     # your Phase 4 result
    n_features   = X_processed.shape[1],
    n_train      = len(X_processed),
    default_rate = float(y.mean()),
    feature_list = shap_importance['feature'].tolist(),
)

with open('model_card.md', 'w') as f:
    f.write(model_card)

print('Model card saved → model_card.md')
print(f'Length: {len(model_card.splitlines())} lines')

---
# Final Summary — All Artefacts Produced

Run this cell last to verify everything was saved correctly.

In [ ]:
import os

artefacts = [
    ('credit_calibrator.joblib',             'Phase 5 — Isotonic calibrator'),
    ('shap_explainer.joblib',                'Phase 6 — SHAP explainer wrapper'),
    ('drift_monitor.joblib',                 'Phase 6 — Drift monitor'),
    ('drift_report.json',                    'Phase 6 — First drift report'),
    ('model_card.md',                        'Phase 6 — Model card'),
    ('phase5_calibrated_submission.csv',     'Phase 5 — Final submission'),
    ('shap_summary.png',                     'Phase 6 — Global SHAP plot'),
    ('shap_waterfall_applicant.png',         'Phase 6 — Per-applicant SHAP'),
    ('drift_psi.png',                        'Phase 6 — Drift PSI chart'),
    ('calibration_comparison.png',           'Phase 5 — Calibration curves'),
]

print(f"{'File':<45} {'Size':>10}  {'Status'}")
print('─' * 70)
all_ok = True
for fname, desc in artefacts:
    exists = os.path.exists(fname)
    size   = f"{os.path.getsize(fname)/1024:.1f} KB" if exists else '—'
    status = '✓' if exists else '✗ MISSING'
    if not exists:
        all_ok = False
    print(f"{fname:<45} {size:>10}  {status}  {desc}")

print('─' * 70)
print('\n✓ All artefacts present — project complete.' if all_ok
      else '\n⚠️  Some artefacts missing — re-run the relevant cells.')

---

## Project Complete — Full Pipeline Summary

| Phase | What was built | Key output |
|---|---|---|
| 1 — Baseline | LightGBM + 5-fold CV on application table | ~0.755 AUC |
| 2 — Feature engineering | 300+ features from 7 relational tables | +0.02 AUC |
| 3 — Tuning + selection | Optuna + SHAP feature pruning | +0.008 AUC |
| 4 — Ensemble | LGB + XGB + CatBoost stacking | **0.7926 AUC** |
| 5 — Calibration | Isotonic regression on OOF scores | True PD probabilities |
| 6 — Production | Drift monitor + SHAP explainer + model card | Deployable system |